# 01 TABULAR FULL PIPELINE
Classification/regression tabular. Edit CFG, run baseline, lalu tambah eksperimen.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
CANDIDATE_SRC = [
    Path('/kaggle/input/datasets/laurentiusfarel/codeset/gemastik_5h_template/src'),
]
for SRC in CANDIDATE_SRC:
    if SRC.exists():
        sys.path = [str(SRC)] + [p for p in sys.path if 'gemastik_5h_template' not in p]
        print('Using SRC:', SRC)
        break
else:
    raise FileNotFoundError('src folder not found. Check Kaggle input path.')

Using SRC: /kaggle/input/datasets/laurentiusfarel/codeset/gemastik_5h_template/src


In [2]:
import numpy as np, pandas as pd
from environment import seed_everything
from common_io import load_competition_tables, detect_id_column, detect_target_column, infer_problem_type
from common_eda import quick_eda
from common_submission import build_submission, validate_submission, save_submission
from common_metrics import class_predictions_from_proba, tune_binary_threshold
from experiment_runner import ExperimentRunner
from tabular_features import prepare_tabular_matrix
from tabular_models import build_tabular_model, available_tabular_models
seed_everything(42)


In [3]:
# ===== CONFIG: WAJIB EDIT/CHECK LINE-BY-LINE =====
CFG = {
    "data_dir": "/kaggle/input",
    "target_col": None,        # isi manual jika auto-detect salah
    "id_col": None,            # isi manual jika auto-detect salah
    "problem_type": "auto",    # classification/regression/auto
    "metric_name": "f1_macro", # contoh: accuracy, f1_macro, log_loss, rmse, mae
    "cv_type": "auto",         # auto/stratified/kfold/group/time
    "group_col": None,
    "n_splits": 5,
    "seed": 42,
}
train, test, sample_submission, paths = load_competition_tables(CFG["data_dir"])
CFG["id_col"] = CFG["id_col"] or detect_id_column(train, test, sample_submission)
CFG["target_col"] = CFG["target_col"] or detect_target_column(train, test, sample_submission)
if CFG["problem_type"] == "auto":
    CFG["problem_type"] = infer_problem_type(train[CFG["target_col"]])
print(CFG)
print(paths)
display(train.head()); display(test.head()); display(sample_submission.head() if sample_submission is not None else None)


{'data_dir': '/kaggle/input', 'target_col': 'target_class', 'id_col': 'row_id', 'problem_type': 'classification', 'metric_name': 'f1_macro', 'cv_type': 'auto', 'group_col': None, 'n_splits': 5, 'seed': 42}
{'train': PosixPath('/kaggle/input/competitions/seleksi-gemastik-2026-problem-1/weather_station_train.csv'), 'test': PosixPath('/kaggle/input/competitions/seleksi-gemastik-2026-problem-1/weather_station_test.csv'), 'sample_submission': None}


,row_id,date,TMP_t_minus_4,TMP_t_minus_3,TMP_t_minus_2,TMP_t_minus_1,TMP_t,DEW_t_minus_4,DEW_t_minus_3,DEW_t_minus_2,...,WND_RATE_t_minus_1,WND_RATE_t,SLP_t_minus_4,SLP_t_minus_3,SLP_t_minus_2,SLP_t_minus_1,SLP_t,latitude,longitude,target_class
0,train_0000000,2014-01-01,-16.1667,-16.9,-17.3667,-17.8333,-18.3,-26.3667,-26.0,-25.8333,...,1.6667,1.0,1020.4000,1020.7,1020.5333,1020.3667,1020.2,48.683333,116.816667,Stable
1,train_0000001,2014-01-01,-17.7333,-17.3,-17.5000,-17.7000,-17.9,-23.2667,-22.7,-22.7000,...,1.0000,1.0,1011.7000,1010.7,1011.5333,1012.3667,1013.2,48.683333,116.816667,Warming
2,train_0000002,2014-01-02,-15.7667,-16.7,-16.0667,-15.4333,-14.8,-20.4333,-21.2,-20.6667,...,3.6667,4.0,1022.9333,1023.9,1024.1333,1024.3667,1024.6,48.683333,116.816667,Cooling
3,train_0000003,2014-01-02,-17.0000,-16.1,-15.5667,-15.0333,-14.5,-19.9333,-18.9,-18.7333,...,2.0000,1.0,1026.6333,1027.0,1027.2667,1027.5333,1027.8,48.683333,116.816667,Stable
4,train_0000004,2014-01-03,-18.1667,-18.5,-18.7667,-19.0333,-19.3,-21.3667,-21.7,-21.8667,...,1.0000,1.0,1028.3000,1028.2,1027.8333,1027.4667,1027.1,48.683333,116.816667,Stable


,row_id,date,TMP_t_minus_4,TMP_t_minus_3,TMP_t_minus_2,TMP_t_minus_1,TMP_t,DEW_t_minus_4,DEW_t_minus_3,DEW_t_minus_2,...,WND_RATE_t_minus_2,WND_RATE_t_minus_1,WND_RATE_t,SLP_t_minus_4,SLP_t_minus_3,SLP_t_minus_2,SLP_t_minus_1,SLP_t,latitude,longitude
0,test_0000000,1/1/2014,19.4,18.9,18.9,17.8,19.4,14.4,15.0,15.0,...,1.5,2.6,2.1,1013.4,1013.4,1013.7,1013.8,1014.2,22.03333,-159.78333
1,test_0000001,1/1/2014,26.1,25.6,24.4,22.8,22.8,18.9,20.0,20.6,...,2.6,2.6,2.1,1011.0,1011.0,1011.7,1012.1,1012.5,22.03333,-159.78333
2,test_0000002,1/2/2014,22.8,22.8,22.8,23.3,23.3,20.6,20.6,21.1,...,2.6,4.1,2.6,1012.6,1012.3,1011.8,1011.4,1010.8,22.03333,-159.78333
3,test_0000003,1/2/2014,25.6,26.1,27.2,26.7,26.1,21.7,21.7,21.7,...,8.2,8.8,7.2,1010.3,1010.2,1009.6,1008.4,1007.3,22.03333,-159.78333
4,test_0000004,1/3/2014,21.0,22.2,21.7,21.1,22.8,20.0,18.9,17.8,...,6.2,5.1,10.3,1008.8,1009.0,1008.6,1008.3,1008.3,22.03333,-159.78333


None

In [4]:
schema, overview = quick_eda(train, test, target_col=CFG["target_col"], id_col=CFG["id_col"])
print("Available models:", available_tabular_models(CFG["problem_type"]))


TRAIN (182567, 25) TEST (204475, 24)
ID: row_id TARGET: target_class
Train duplicates: 0 Test duplicates: 0
Schema: {'numeric': 22, 'categorical': 0, 'datetime': 1, 'text': 0, 'pathlike': 0}
Target summary:


,target,count,pct
0,Stable,75978,0.416165
1,Cooling,62356,0.341551
2,Warming,44233,0.242284


,table,column,dtype,missing,missing_pct,nunique,sample_values
0,train,row_id,object,0,0.0,182567,"train_0000000, train_0000001, train_0000002"
1,train,date,object,0,0.0,3652,"2014-01-01, 2014-01-01, 2014-01-02"
2,train,TMP_t_minus_4,float64,0,0.0,4550,"-16.1667, -17.7333, -15.7667"
3,train,TMP_t_minus_3,float64,0,0.0,3075,"-16.9, -17.3, -16.7"
4,train,TMP_t_minus_2,float64,0,0.0,5751,"-17.3667, -17.5, -16.0667"
5,train,TMP_t_minus_1,float64,0,0.0,5687,"-17.8333, -17.7, -15.4333"
6,train,TMP_t,float64,0,0.0,3087,"-18.3, -17.9, -14.8"
7,train,DEW_t_minus_4,float64,0,0.0,4489,"-26.3667, -23.2667, -20.4333"
8,train,DEW_t_minus_3,float64,0,0.0,3028,"-26.0, -22.7, -21.2"
9,train,DEW_t_minus_2,float64,0,0.0,5778,"-25.8333, -22.7, -20.6667"


Available models: ['histgbm', 'rf', 'extra', 'logreg', 'ridge_cls', 'lgbm', 'xgb', 'catboost']


In [5]:
# Manual cleaning hook. Tulis cleaning khusus di sini, jangan ubah fungsi src dulu.
def manual_clean(train, test):
    train = train.copy(); test = test.copy()
    # contoh:
    # for c in ["leaky_col", "debug_col"]:
    #     if c in train.columns: train = train.drop(columns=[c])
    #     if c in test.columns: test = test.drop(columns=[c])
    return train, test

train_c, test_c = manual_clean(train, test)


In [6]:
# ===== EXPERIMENT QUEUE =====
EXPERIMENTS = [
    {"name": "tab_histgbm_basic", "features": {"missing_indicators": True, "frequency_encoding": True, "numeric_row_features": True, "onehot_max_categories": 0, "force_dense": True}, "model": {"name": "histgbm", "max_iter": 250}},
    {"name": "tab_extra_basic", "features": {"missing_indicators": True, "frequency_encoding": True}, "model": {"name": "extra", "n_estimators": 400, "min_samples_leaf": 2}},
    # uncomment jika tersedia dan waktu cukup
    # {"name": "tab_lgbm_basic", "features": {"missing_indicators": True, "frequency_encoding": True, "numeric_row_features": True}, "model": {"name": "lgbm", "n_estimators": 800, "learning_rate": 0.03}},
]


In [7]:
runner = ExperimentRunner(output_dir="/kaggle/working/experiments_tabular", metric_name=CFG["metric_name"], problem_type=CFG["problem_type"], seed=CFG["seed"], n_splits=CFG["n_splits"])
last_payload = None
for exp in EXPERIMENTS:
    print("\n===", exp["name"], "===")
    X, y, X_test, info = prepare_tabular_matrix(train_c, test_c, CFG["target_col"], CFG["id_col"], exp.get("features", {}))
    model_cfg = exp.get("model", {}).copy()
    model_name = model_cfg.pop("name")
    model = build_tabular_model(model_name, problem_type=CFG["problem_type"], seed=CFG["seed"], **model_cfg)
    groups = train_c[CFG["group_col"]].values if CFG.get("group_col") else None
    result, oof, test_pred, le = runner.run_matrix_experiment(exp["name"], X, y, X_test, model, cv_type=CFG["cv_type"], groups=groups, notes=str(exp))
    last_payload = (exp, result, oof, test_pred, le)

display(runner.leaderboard())



=== tab_histgbm_basic ===
tab_histgbm_basic fold 0: 0.709102
tab_histgbm_basic fold 1: 0.705792
tab_histgbm_basic fold 2: 0.706096
tab_histgbm_basic fold 3: 0.700899
tab_histgbm_basic fold 4: 0.702960
DONE tab_histgbm_basic: 0.704970 ± 0.002815 in 177.1s

=== tab_extra_basic ===
tab_extra_basic fold 0: 0.674146
tab_extra_basic fold 1: 0.672200
tab_extra_basic fold 2: 0.671546
tab_extra_basic fold 3: 0.670254
tab_extra_basic fold 4: 0.670368
DONE tab_extra_basic: 0.671703 ± 0.001423 in 290.7s


,name,score,std,seconds,notes,path_oof,path_test
0,tab_histgbm_basic,0.704970,0.002815,177.098919,"{'name': 'tab_histgbm_basic', 'features': {'mi...",/kaggle/working/experiments_tabular/tab_histgb...,/kaggle/working/experiments_tabular/tab_histgb...
1,tab_extra_basic,0.671703,0.001423,290.650129,"{'name': 'tab_extra_basic', 'features': {'miss...",/kaggle/working/experiments_tabular/tab_extra_...,/kaggle/working/experiments_tabular/tab_extra_...


In [8]:
# Build best submission from selected experiment.
lb = runner.leaderboard()
best = lb.iloc[0]
test_pred = np.load(best.path_test, allow_pickle=True)
if CFG["problem_type"] == "classification":
    pred = class_predictions_from_proba(test_pred)
    # Kalau label asli string, inverse transform dari last run jika sesuai. Cek manual.
    try:
        if last_payload[-1] is not None:
            pred = last_payload[-1].inverse_transform(pred.astype(int))
    except Exception as e:
        print("inverse_transform skipped", e)
else:
    pred = test_pred.reshape(-1)
sub = build_submission(test_c, pred, sample_submission=sample_submission, id_col=CFG["id_col"], target_col=CFG["target_col"])
validate_submission(sub, sample_submission)
save_submission(sub, "/kaggle/working/submission_tabular.csv")
display(sub.head())


Submission OK: (204475, 2)
Saved submission: /kaggle/working/submission_tabular.csv


,row_id,target_class
0,test_0000000,Cooling
1,test_0000001,Cooling
2,test_0000002,Stable
3,test_0000003,Cooling
4,test_0000004,Cooling
